# 01 — Data Validation

**Purpose:** Validate all input data, check referential integrity, report quality.
**Inputs:** All parquet files in `data/processed/`
**Outputs:** Data quality summary; optionally cleaned intermediates.

In [ ]:
from __future__ import annotations

import json
import warnings
from pathlib import Path

import pandas as pd
import pandera as pa

from buildanalysis.loading import _SCHEMA_MAP, BuildDataset

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_DIR = Path("../../data/processed")
INTERMEDIATE_DIR = DATA_DIR / "intermediate"

ds = BuildDataset(DATA_DIR, intermediate_dir=INTERMEDIATE_DIR, validate=False)

# Core files (must exist for analysis to proceed)
CORE_FILES = ["file_metrics", "target_metrics", "edge_list", "contributor_target_commits"]

# Stage 0 files (optional — produced by supplementary collection scripts)
STAGE0_FILES = ["git_commit_log", "header_edges", "header_metrics", "build_schedule"]

ALL_FILES = CORE_FILES + STAGE0_FILES

print("Data directory:", DATA_DIR.resolve())
print("Intermediate directory:", INTERMEDIATE_DIR.resolve())

Data directory: /Users/cgilbert/Work/projects/build-optimisation/notebooks/data/processed
Intermediate directory: /Users/cgilbert/Work/projects/build-optimisation/notebooks/data/processed/intermediate


/Users/cgilbert/Work/projects/build-optimisation/.venv/lib/python3.13/site-packages/pandera/_pandas_deprecated.py:154: FutureWarning: Importing pandas-specific classes and functions from the
top-level pandera module will be **removed in a future version of pandera**.
If you're using pandera to validate pandas objects, we highly recommend updating
your import:

```
# old import
import pandera as pa

# new import
import pandera.pandas as pa
```

If you're using pandera to validate objects from other compatible libraries
like pyspark or polars, see the supported libraries section of the documentation
for more information on how to import pandera:

https://pandera.readthedocs.io/en/stable/supported_libraries.html

To disable this warning, set the environment variable:

```
export DISABLE_PANDERA_IMPORT_WARNING=True
```

  warnings.warn(_future_warning, FutureWarning)


## 1. Load All Datasets

Check which parquet files exist and report their shape.

In [2]:
loaded: dict[str, pd.DataFrame] = {}
file_status: dict[str, dict] = {}

for name in ALL_FILES:
    exists = ds.has_file(name)
    status = {"exists": exists, "rows": None, "cols": None}
    if exists:
        try:
            df = pd.read_parquet(DATA_DIR / f"{name}.parquet")
            loaded[name] = df
            status["rows"] = len(df)
            status["cols"] = len(df.columns)
        except Exception as e:
            status["error"] = str(e)
    file_status[name] = status

# Report
core_present = sum(1 for n in CORE_FILES if file_status[n]["exists"])
stage0_present = sum(1 for n in STAGE0_FILES if file_status[n]["exists"])
stage0_missing = [n for n in STAGE0_FILES if not file_status[n]["exists"]]

print("FILE INVENTORY")
print("=" * 60)
print(f"{'File':<35} {'Exists':<8} {'Rows':<10} {'Cols':<6}")
print("-" * 60)
for name in ALL_FILES:
    s = file_status[name]
    exists_str = "Y" if s["exists"] else "N"
    rows_str = f"{s['rows']:,}" if s["rows"] is not None else "-"
    cols_str = str(s["cols"]) if s["cols"] is not None else "-"
    print(f"{name:<35} {exists_str:<8} {rows_str:<10} {cols_str:<6}")
print("-" * 60)
print(f"Core files: {core_present}/{len(CORE_FILES)} present")
print(f"Stage 0 files: {stage0_present}/{len(STAGE0_FILES)} present", end="")
if stage0_missing:
    print(f" (missing: {', '.join(stage0_missing)})")
else:
    print()

FILE INVENTORY
File                                Exists   Rows       Cols  
------------------------------------------------------------
file_metrics                        N        -          -     
target_metrics                      N        -          -     
edge_list                           N        -          -     
contributor_target_commits          N        -          -     
git_commit_log                      N        -          -     
header_edges                        N        -          -     
header_metrics                      N        -          -     
build_schedule                      N        -          -     
------------------------------------------------------------
Core files: 0/4 present
Stage 0 files: 0/4 present (missing: git_commit_log, header_edges, header_metrics, build_schedule)


## 2. Schema Validation

Validate each loaded file against its Pandera schema. Report pass/fail with error details.

In [3]:
schema_results: dict[str, dict] = {}

print("SCHEMA VALIDATION")
print("=" * 60)

for name in ALL_FILES:
    if name not in loaded:
        schema_results[name] = {"status": "SKIPPED", "reason": "file not loaded"}
        print(f"  {name:<35} SKIPPED (file not loaded)")
        continue

    if name not in _SCHEMA_MAP:
        schema_results[name] = {"status": "SKIPPED", "reason": "no schema defined"}
        print(f"  {name:<35} SKIPPED (no schema)")
        continue

    schema_cls = _SCHEMA_MAP[name]
    try:
        schema_cls.validate(loaded[name])
        schema_results[name] = {"status": "PASSED"}
        print(f"  {name:<35} PASSED")
    except pa.errors.SchemaError as e:
        schema_results[name] = {"status": "FAILED", "error": str(e)}
        print(f"  {name:<35} FAILED")
        # Print first few lines of the error
        for line in str(e).splitlines()[:5]:
            print(f"    {line}")

print("-" * 60)
passed = sum(1 for v in schema_results.values() if v["status"] == "PASSED")
failed = sum(1 for v in schema_results.values() if v["status"] == "FAILED")
skipped = sum(1 for v in schema_results.values() if v["status"] == "SKIPPED")
print(f"Passed: {passed}  Failed: {failed}  Skipped: {skipped}")

SCHEMA VALIDATION
  file_metrics                        SKIPPED (file not loaded)
  target_metrics                      SKIPPED (file not loaded)
  edge_list                           SKIPPED (file not loaded)
  contributor_target_commits          SKIPPED (file not loaded)
  git_commit_log                      SKIPPED (file not loaded)
  header_edges                        SKIPPED (file not loaded)
  header_metrics                      SKIPPED (file not loaded)
  build_schedule                      SKIPPED (file not loaded)
------------------------------------------------------------
Passed: 0  Failed: 0  Skipped: 8


## 3. Referential Integrity Checks

Verify that foreign-key-like relationships hold across datasets.

In [4]:
integrity_issues: list[str] = []


def check_fk(
    child_name: str,
    child_col: str,
    parent_name: str,
    parent_col: str,
    label: str | None = None,
) -> None:
    """Check that all values in child_col exist in parent_col. Report results."""
    label = label or f"{child_name}.{child_col} -> {parent_name}.{parent_col}"

    if child_name not in loaded or parent_name not in loaded:
        print(f"  {label}: SKIPPED (missing data)")
        return

    child_vals = set(loaded[child_name][child_col].dropna().unique())
    parent_vals = set(loaded[parent_name][parent_col].dropna().unique())
    missing = child_vals - parent_vals

    total = len(child_vals)
    n_ok = total - len(missing)

    if missing:
        # Count rows affected, not just unique values
        rows_affected = loaded[child_name][child_col].isin(missing).sum()
        msg = f"{rows_affected} {child_name} rows reference unknown {parent_col} values"
        integrity_issues.append(msg)
        print(f"  {label}")
        print(f"    Checked: {total}  Passing: {n_ok}  Failing: {len(missing)}  Rows affected: {rows_affected}")
        examples = sorted(missing)[:5]
        print(f"    Examples: {examples}")
    else:
        print(f"  {label}")
        print(f"    Checked: {total}  Passing: {n_ok}  Failing: 0  PASSED")


print("REFERENTIAL INTEGRITY")
print("=" * 60)

# Core checks
check_fk("file_metrics", "cmake_target", "target_metrics", "cmake_target")
check_fk("edge_list", "source_target", "target_metrics", "cmake_target")
check_fk("edge_list", "dest_target", "target_metrics", "cmake_target")
check_fk("contributor_target_commits", "cmake_target", "target_metrics", "cmake_target")

# Stage 0 checks (spot-checks)
if "git_commit_log" in loaded and "file_metrics" in loaded:
    git_files = set(loaded["git_commit_log"]["source_file"].dropna().unique())
    fm_files = set(loaded["file_metrics"]["source_file"].dropna().unique())
    overlap = git_files & fm_files
    pct = 100 * len(overlap) / len(git_files) if git_files else 0
    print("  git_commit_log.source_file vs file_metrics.source_file")
    print(f"    git_commit_log files: {len(git_files)}  overlap with file_metrics: {len(overlap)} ({pct:.1f}%)")
    if pct < 50:
        integrity_issues.append(f"git_commit_log has low file overlap with file_metrics ({pct:.1f}%)")

if "header_edges" in loaded and "file_metrics" in loaded:
    he_files = set(loaded["header_edges"]["source_file"].dropna().unique())
    fm_files = set(loaded["file_metrics"]["source_file"].dropna().unique())
    overlap = he_files & fm_files
    pct = 100 * len(overlap) / len(he_files) if he_files else 0
    print("  header_edges.source_file vs file_metrics.source_file")
    print(f"    header_edges files: {len(he_files)}  overlap with file_metrics: {len(overlap)} ({pct:.1f}%)")
    if pct < 50:
        integrity_issues.append(f"header_edges has low file overlap with file_metrics ({pct:.1f}%)")

if "build_schedule" in loaded and "target_metrics" in loaded:
    check_fk("build_schedule", "cmake_target", "target_metrics", "cmake_target")

print("-" * 60)
if integrity_issues:
    print(f"{len(integrity_issues)} integrity issue(s) found:")
    for issue in integrity_issues:
        print(f"  - {issue}")
else:
    print("All referential integrity checks passed")

REFERENTIAL INTEGRITY
  file_metrics.cmake_target -> target_metrics.cmake_target: SKIPPED (missing data)
  edge_list.source_target -> target_metrics.cmake_target: SKIPPED (missing data)
  edge_list.dest_target -> target_metrics.cmake_target: SKIPPED (missing data)
  contributor_target_commits.cmake_target -> target_metrics.cmake_target: SKIPPED (missing data)
------------------------------------------------------------
All referential integrity checks passed


## 4. Null Analysis

Report null counts and percentages per column for each file. Flag columns with >10% nulls.

In [5]:
null_flags: list[str] = []

print("NULL ANALYSIS")
print("=" * 60)

for name in ALL_FILES:
    if name not in loaded:
        continue

    df = loaded[name]
    null_counts = df.isnull().sum()
    null_pct = 100 * null_counts / len(df)
    has_nulls = null_counts[null_counts > 0]

    print(f"\n--- {name} ({len(df):,} rows) ---")
    if has_nulls.empty:
        print("  No null values")
        continue

    report = pd.DataFrame({
        "null_count": has_nulls,
        "null_pct": null_pct[has_nulls.index].round(1),
    }).sort_values("null_pct", ascending=False)

    for col, row in report.iterrows():
        flag = " <<<" if row["null_pct"] > 10 else ""
        print(f"  {col:<40} {int(row['null_count']):>8} nulls  ({row['null_pct']:>5.1f}%){flag}")
        if row["null_pct"] > 10:
            null_flags.append(f"{name}.{col}: {row['null_pct']:.0f}% null")

print("\n" + "=" * 60)
if null_flags:
    print(f"{len(null_flags)} column(s) with >10% nulls:")
    for flag in null_flags:
        print(f"  - {flag}")
else:
    print("No columns with >10% nulls")

NULL ANALYSIS

No columns with >10% nulls


## 5. Distribution Sanity Checks

Verify that key numeric columns have plausible ranges and detect zero-variance columns.

In [6]:
distribution_issues: list[str] = []

# Range checks: (file_name, column, min_ok, max_ok, description)
RANGE_CHECKS = [
    ("file_metrics", "compile_time_ms", 0, 600_000, "compile time (ms), expect < 10 min"),
    ("file_metrics", "code_lines", 0, 100_000, "lines of code, expect < 100k"),
    ("file_metrics", "preprocessed_bytes", 0, None, "preprocessed size (bytes), expect non-negative"),
    ("target_metrics", "total_build_time_ms", 0, None, "total build time (ms), expect non-negative"),
    ("target_metrics", "betweenness_centrality", 0, 1, "betweenness centrality, expect [0, 1]"),
]

print("DISTRIBUTION SANITY CHECKS")
print("=" * 60)

for file_name, col, lo, hi, desc in RANGE_CHECKS:
    if file_name not in loaded or col not in loaded[file_name].columns:
        print(f"  {file_name}.{col}: SKIPPED (not available)")
        continue

    series = loaded[file_name][col].dropna()
    if series.empty:
        print(f"  {file_name}.{col}: SKIPPED (all null)")
        continue

    s_min = series.min()
    s_max = series.max()
    s_mean = series.mean()

    violations = 0
    if lo is not None:
        violations += (series < lo).sum()
    if hi is not None:
        violations += (series > hi).sum()

    status = "PASSED" if violations == 0 else f"FAILED ({violations} violations)"
    if violations > 0:
        distribution_issues.append(f"{file_name}.{col}: {violations} values outside [{lo}, {hi}]")

    print(f"  {file_name}.{col} — {desc}")
    print(f"    min={s_min:.2f}  max={s_max:.2f}  mean={s_mean:.2f}  violations={violations}  {status}")

# Zero-variance check
print("\n--- Zero-variance columns ---")
zero_var_found = False
for name in ALL_FILES:
    if name not in loaded:
        continue
    df = loaded[name]
    numeric_cols = df.select_dtypes(include="number").columns
    for col in numeric_cols:
        non_null = df[col].dropna()
        if len(non_null) > 1 and non_null.nunique() == 1:
            print(f"  {name}.{col} = {non_null.iloc[0]} (constant)")
            zero_var_found = True

if not zero_var_found:
    print("  No zero-variance numeric columns found")

print("\n" + "=" * 60)
if distribution_issues:
    print(f"{len(distribution_issues)} distribution issue(s):")
    for issue in distribution_issues:
        print(f"  - {issue}")
else:
    print("All distribution checks passed")

DISTRIBUTION SANITY CHECKS
  file_metrics.compile_time_ms: SKIPPED (not available)
  file_metrics.code_lines: SKIPPED (not available)
  file_metrics.preprocessed_bytes: SKIPPED (not available)
  target_metrics.total_build_time_ms: SKIPPED (not available)
  target_metrics.betweenness_centrality: SKIPPED (not available)

--- Zero-variance columns ---
  No zero-variance numeric columns found

All distribution checks passed


## 6. Data Quality Summary

In [7]:
# Build summary table
summary_rows = []
for name in ALL_FILES:
    s = file_status[name]
    schema_s = schema_results.get(name, {}).get("status", "N/A")
    row_count = s["rows"] if s["rows"] is not None else "-"

    # Count null columns for this file
    null_cols = 0
    if name in loaded:
        null_cols = (loaded[name].isnull().sum() > 0).sum()

    # Count integrity issues mentioning this file
    n_integrity = sum(1 for i in integrity_issues if name in i)

    summary_rows.append({
        "File": name,
        "Exists": "Y" if s["exists"] else "N",
        "Schema": schema_s,
        "Rows": row_count,
        "Null Cols": null_cols if name in loaded else "-",
        "Integrity Issues": n_integrity,
    })

summary_df = pd.DataFrame(summary_rows)
print("DATA QUALITY SUMMARY TABLE")
print("=" * 60)
print(summary_df.to_string(index=False))

DATA QUALITY SUMMARY TABLE
                      File Exists  Schema Rows Null Cols  Integrity Issues
              file_metrics      N SKIPPED    -         -                 0
            target_metrics      N SKIPPED    -         -                 0
                 edge_list      N SKIPPED    -         -                 0
contributor_target_commits      N SKIPPED    -         -                 0
            git_commit_log      N SKIPPED    -         -                 0
              header_edges      N SKIPPED    -         -                 0
            header_metrics      N SKIPPED    -         -                 0
            build_schedule      N SKIPPED    -         -                 0


## 7. Extract Header Include Edges (Fallback)

If `header_edges.parquet` doesn't exist but `file_metrics.header_tree` contains JSON blobs,
parse them into a header edge list and save as an intermediate file.

In [8]:
if "header_edges" in loaded:
    print("header_edges.parquet already exists — skipping fallback extraction.")
elif "file_metrics" in loaded and "header_tree" in loaded["file_metrics"].columns:
    fm = loaded["file_metrics"]
    trees = fm[["source_file", "header_tree"]].dropna(subset=["header_tree"])
    print(f"Parsing header_tree from {len(trees)} file_metrics rows...")

    edge_rows: list[dict] = []

    def walk_tree(node: dict, source_file: str, depth: int = 0) -> None:
        """Recursively walk a header tree node and collect edges."""
        name = node.get("name", "")
        children = node.get("children", [])
        for child in children:
            child_name = child.get("name", "")
            child_is_system = child_name.startswith("<") or "/usr/" in child_name
            edge_rows.append({
                "includer": name,
                "included": child_name,
                "depth": depth + 1,
                "source_file": source_file,
                "is_system": child_is_system,
            })
            walk_tree(child, source_file, depth + 1)

    parse_errors = 0
    for _, row in trees.iterrows():
        try:
            tree = json.loads(row["header_tree"])
            if isinstance(tree, dict):
                walk_tree(tree, row["source_file"])
            elif isinstance(tree, list):
                for node in tree:
                    walk_tree(node, row["source_file"])
        except (json.JSONDecodeError, TypeError):
            parse_errors += 1

    if edge_rows:
        header_edges_df = pd.DataFrame(edge_rows)
        INTERMEDIATE_DIR.mkdir(parents=True, exist_ok=True)
        out_path = INTERMEDIATE_DIR / "header_edges_from_tree.parquet"
        header_edges_df.to_parquet(out_path, index=False)
        print(f"Extracted {len(header_edges_df):,} header edges -> {out_path}")
        if parse_errors:
            print(f"  ({parse_errors} rows had unparseable header_tree JSON)")
    else:
        print("No header edges extracted (header_tree blobs were empty or unparseable)")
        if parse_errors:
            print(f"  ({parse_errors} parse errors)")
else:
    print("Neither header_edges.parquet nor file_metrics.header_tree available — skipping.")

Neither header_edges.parquet nor file_metrics.header_tree available — skipping.


## 8. Assessment

In [9]:
print()
print("DATA VALIDATION SUMMARY")
print("=======================")

# Core files
core_ok = all(file_status[n]["exists"] for n in CORE_FILES)
core_str = f"{core_present}/{len(CORE_FILES)} present"
print(f"Core files: {core_str} {'✓' if core_ok else '✗ BLOCKING'}")

# Stage 0 files
stage0_str = f"{stage0_present}/{len(STAGE0_FILES)} present"
if stage0_missing:
    stage0_str += f" (missing: {', '.join(stage0_missing)})"
print(f"Stage 0 files: {stage0_str}")

# Schema validation
schema_passed = all(
    schema_results.get(n, {}).get("status") in ("PASSED", "SKIPPED")
    for n in ALL_FILES
)
schema_failed_names = [
    n for n in ALL_FILES
    if schema_results.get(n, {}).get("status") == "FAILED"
]
if schema_passed:
    print("Schema validation: All passed ✓")
else:
    print(f"Schema validation: {len(schema_failed_names)} FAILED ✗")
    for n in schema_failed_names:
        print(f"  - {n}")

# Referential integrity
if integrity_issues:
    print(f"Referential integrity: {len(integrity_issues)} issue(s) found ⚠")
    for issue in integrity_issues:
        print(f"  - {issue}")
else:
    print("Referential integrity: All passed ✓")

# Null analysis
if null_flags:
    print(f"Null analysis: {len(null_flags)} column(s) with >10% nulls ⚠")
    for flag in null_flags:
        print(f"  - {flag}")
else:
    print("Null analysis: All passed ✓")

# Distribution checks
if distribution_issues:
    print(f"Distribution checks: {len(distribution_issues)} issue(s) ⚠")
    for issue in distribution_issues:
        print(f"  - {issue}")
else:
    print("Distribution checks: All passed ✓")

# Overall assessment
print()
blocking = not core_ok or not schema_passed
if blocking:
    print("ASSESSMENT: BLOCKING issues found. Fix before proceeding with analysis.")
elif integrity_issues or null_flags or distribution_issues:
    print("ASSESSMENT: Proceed with analysis. Minor issues are non-blocking.")
else:
    print("ASSESSMENT: All checks passed. Data is ready for analysis.")


DATA VALIDATION SUMMARY
Core files: 0/4 present ✗ BLOCKING
Stage 0 files: 0/4 present (missing: git_commit_log, header_edges, header_metrics, build_schedule)
Schema validation: All passed ✓
Referential integrity: All passed ✓
Null analysis: All passed ✓
Distribution checks: All passed ✓

ASSESSMENT: BLOCKING issues found. Fix before proceeding with analysis.
